# GIAO THỨC CHẠY LẠI THỰC NGHIỆM ĐỐI SÁNH CEGR VÀ CÁC BIẾN THỂ ABLATION
## ÁP DỤNG TRÊN CẢ 3 MÔI TRƯỜNG DOORKEY: 5x5, 6x6, 8x8 (ICAI-FAI 2026)

Notebook này được thiết kế theo đúng quy chuẩn nghiêm ngặt tại **`CEGR_rerun_logic_system.md`**:
1. **Mục tiêu độc quyền:** Chỉ chạy đối sánh **CEGR** và **các biến thể Ablation** (`noClass`, `noFeedback`, `noPool`, `noAnchors`, `noG2`, `noG4`, `noG5`, cùng mốc `round0`). Loại bỏ các baseline bên ngoài để tập trung ngân sách và phân tích chuyên sâu.
2. **Một đợt chạy duy nhất & Cùng Harness:** CEGR và toàn bộ Ablation trên cả 3 môi trường (`doorkey5`, `doorkey6`, `doorkey8`) cùng chạy trong 1 đợt, cùng commit mã nguồn, cùng phiên bản vLLM.
3. **Tập Ablation nâng cấp $N=20$:** Tuyển chọn tập con $N=20$ base reward cố định bằng seed `20261031`, xuất file `selection_manifest.json` bất biến.
4. **Replicate & Noise Floor:** Chạy $R=3$ replicate độc lập cho CEGR để tính ngưỡng nhiễu tự nhiên (*noise floor*), làm mốc tham chiếu cho mọi $\Delta \text{VF}$.
5. **Kiểm tra tự động & Thống kê chuẩn:** Sanity check 12 điều kiện trước khi tính toán, Paired Permutation Test chính xác ($2^n$ hoán vị dấu với $N=20$), Bootstrap CI 10,000 lần và hiệu chỉnh Holm cho cả họ chính (H1) lẫn họ thăm dò.

In [ ]:
# =====================================================================
# CELL 0: THIẾT LẬP MÔI TRƯỜNG & CLONE REPO (KAGGLE / COLAB)
# =====================================================================
import os
import sys

# Tự động clone kho mã nguồn CEGR nếu chạy trên Colab / Kaggle và chưa có mã nguồn
if not os.path.exists("cegr") and not os.path.exists("../cegr"):
    if not os.path.exists("CEGR"):
        print(">> Đang clone kho mã nguồn CEGR từ GitHub...")
        !git clone https://github.com/ToanThinh1208/CEGR.git
    else:
        print(">> Thư mục CEGR đã tồn tại trên đĩa.")
    
    if os.path.exists("CEGR"):
        %cd CEGR
elif os.path.exists("CEGR") and not os.path.exists("cegr"):
    %cd CEGR

repo_root = os.path.abspath(".") if os.path.exists("cegr") else (os.path.abspath("..") if os.path.exists("../cegr") else os.path.abspath("CEGR"))
for p in [repo_root, os.path.abspath("."), os.path.abspath("..")]:
    if os.path.exists(os.path.join(p, "cegr")) and p not in sys.path:
        sys.path.insert(0, p)

print(f"✅ Thư mục làm việc hiện tại: {os.getcwd()}")
print(f"✅ Repo root trong sys.path: {repo_root}")

try:
    import cegr
    print(f"✅ Đã sẵn sàng nạp package 'cegr' tại: {cegr.__file__}")
except ImportError as e:
    print(f"⚠️ Lưu ý: Chưa thể nạp package cegr ({e}).")


In [ ]:
# =====================================================================
# CELL 1: CÀI ĐẶT THƯ VIỆN & CẤU HÌNH TOÀN CỤC CHUẨN HÓA
# =====================================================================
import subprocess, sys, os

# Gỡ các gói xung đột và cài Torch + vLLM trên Colab / Kaggle
# !pip uninstall -y torchvision torchaudio cudf-cu12 cuml-cu12 dask-cuda pylibcudf-cu12 > /dev/null 2>&1
# !pip install -q torch==2.6.0+cu124 --index-url https://download.pytorch.org/whl/cu124
# !pip install -q vllm transformers accelerate tiktoken autoawq hf_transfer

for p in [os.path.abspath('..'), os.path.abspath('.')]:
    if p not in sys.path: sys.path.insert(0, p)

import json, time, random, collections, hashlib, math, re, ast, itertools
from typing import Dict, List, Any, Optional, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi": 130, "savefig.dpi": 300,
    "axes.grid": True, "grid.alpha": 0.25,
    "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10
})

# ============== CÔNG TẮC & NGÂN SÁCH THỰC NGHIỆM ==============
IS_COLAB          = False      # True: Colab (/content/cegr_out) | False: Kaggle (/kaggle/working/cegr)
FAST_SMOKE_TEST   = False      # True: chạy thử 1-2 mẫu kiểm tra toàn bộ pipeline | False: chạy thật đầy đủ

# 3 Môi trường DoorKey liên tục
ENV_NAMES         = ["doorkey5", "doorkey6", "doorkey8"]

# Kích thước tập Ablation ABL: Nâng cấp lên n = 20 base rewards
N_ABL_REWARDS     = 2 if FAST_SMOKE_TEST else 20
N_SAMPLES_PER_JOB = 1 if FAST_SMOKE_TEST else 3    # 3 sample_id (0, 1, 2) cho mỗi base reward
N_CEGR_REPLICATES = 1 if FAST_SMOKE_TEST else 3    # R = 3 cho CEGR để tính noise floor
SEED_SELECT_ABL   = 20261031                       # Seed cố định chọn tập ABL theo mục 1.2

# Mô hình LLM & Tham số sinh
MODEL_ID          = "Qwen/Qwen2.5-Coder-7B-Instruct-AWQ"
MAX_MODEL_LEN     = 6144
N_ROUNDS          = 4
REPAIR_TEMP       = 0.35
REPAIR_MAXTOK     = 900

# Danh sách các Arm thực nghiệm (Theo mục 1.3 của giao thức)
# CEGR chạy R replicate; Các ablation chạy 1 replicate (hoặc 2 nếu đủ compute)
ABLATION_ARMS     = [
    "CEGR_noClass",     # Primary H1: Gộp nhãn vào generic / _flat
    "CEGR_noFeedback",  # New Baseline: Chỉ resample lại cùng ngân sách, không feedback CE/rule/anchor
    "CEGR_noPool",      # Tắt CE pool
    "CEGR_noAnchors",   # Tắt cập nhật dynamic anchor
    "CEGR_noG2",        # Bỏ Gate 2 (AST)
    "CEGR_noG4",        # Bỏ Gate 4 (Spearman)
    "CEGR_noG5",        # Bỏ Gate 5 (Short RL probe)
]

# Cấu hình GateKeeper, Seed streams & Scoring
SCORING_PROFILE   = "relaxed"
KEYPOS_MODE       = "agent_pos"
CE_STREAM_SEED    = 1000
SIGNAL_STREAM_SEED= 2000
EVAL_STREAM_SEED  = 3000

# Thư mục lưu kết quả
OUT_DIR = "/content/cegr_out/rerun_ablations" if IS_COLAB else "/kaggle/working/cegr/rerun_ablations"
if not os.path.exists(OUT_DIR) and not IS_COLAB and not os.path.exists("/kaggle"):
    OUT_DIR = os.path.abspath("./scratch_test_out/rerun_ablations")
os.makedirs(OUT_DIR, exist_ok=True)

print(f"✓ Cấu hình thực nghiệm sẵn sàng:")
print(f"  -> Môi trường: {ENV_NAMES}")
print(f"  -> Quy mô ABL: N = {N_ABL_REWARDS} base rewards/env (3 samples/job, CEGR R={N_CEGR_REPLICATES})")
print(f"  -> Các Arm: CEGR (baseline) + {ABLATION_ARMS}")
print(f"  -> Thư mục kết quả: {OUT_DIR}")


In [ ]:
# =====================================================================
# CELL 2: KHỞI TẠO 3 MÔI TRƯỜNG & KITS THAM CHIẾU TÁCH BIỆT
# =====================================================================
from cegr.envs.doorkey import DoorKeyEnv, make_env, ENV_CONFIGS
from cegr.synthesis.runner import Kit, build_kit

envs = {}
kits = {}

for env_name in ENV_NAMES:
    e = make_env(env_name)
    k = build_kit(e, ce_seed=CE_STREAM_SEED, signal_seed=SIGNAL_STREAM_SEED, eval_seed=EVAL_STREAM_SEED)
    envs[env_name] = e
    kits[env_name] = k
    print(f"✓ Đã nạp môi trường {env_name}: {e.n} states khả đạt, Horizon=(T1={e.horizon}, T2={2*e.horizon})")
    print(f"   Kit: {len(k.p0)} cặp P0, {len(k.signal_trajs)} Signal trajs, {len(k.eval_trajs)} Eval trajs.")


In [ ]:
# =====================================================================
# CELL 3: TUYỂN CHỌN TẬP ABL ĐỘC LẬP & XUẤT MANIFEST BẤT BIẾN
# =====================================================================
from cegr.synthesis.dataset import build_calibrated_mutants

manifest_file = os.path.join(OUT_DIR, "selection_manifest.json")
abl_manifest = {}

if os.path.exists(manifest_file):
    with open(manifest_file, "r", encoding="utf-8") as f:
        abl_manifest = json.load(f)
    print(f"✓ Nạp manifest ABL có sẵn ({len(abl_manifest)} môi trường) từ: {manifest_file}")
else:
    print(f">>> Đang tạo mới selection_manifest.json cho {ENV_NAMES} với seed={SEED_SELECT_ABL}...")
    for env_name in ENV_NAMES:
        e = envs[env_name]
        # Sinh pool các mutant chuẩn hóa qua 12 họ
        mut_pool = build_calibrated_mutants(e)
        
        # Rút mẫu ngẫu nhiên không hoàn lại N_ABL_REWARDS bằng seed_select cố định
        rng = random.Random(SEED_SELECT_ABL)
        rng.shuffle(mut_pool)
        selected = mut_pool[:N_ABL_REWARDS]
        
        # Gán mã nhận diện rõ ràng cho base reward
        for idx, item in enumerate(selected):
            item["base_id"] = f"{env_name}__abl_{idx:03d}__{item['id']}"
            item["env"] = env_name
        abl_manifest[env_name] = selected
        print(f"  -> {env_name}: Đã chọn {len(selected)} base rewards cho tập ABL.")
        
    # Ghi manifest cố định không sửa đổi
    manifest_content = json.dumps(abl_manifest, indent=2, ensure_ascii=False)
    content_hash = hashlib.sha256(manifest_content.encode('utf-8')).hexdigest()
    manifest_meta = {
        "seed_select": SEED_SELECT_ABL,
        "n_per_env": N_ABL_REWARDS,
        "environments": ENV_NAMES,
        "sha256": content_hash,
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "manifest": abl_manifest,
    }
    with open(manifest_file, "w", encoding="utf-8") as f:
        json.dump(manifest_meta, f, indent=2, ensure_ascii=False)
    print(f"✓ Đã ghi lưu manifest bất biến (SHA256: {content_hash[:12]}...) tại {manifest_file}")


In [ ]:
# =====================================================================
# CELL 4: KHỞI TẠO ĐỘNG CƠ vLLM BATCH INFERENCE
# =====================================================================
from cegr.engine.llm_vllm import init_llm, llm_batch_chat, extract_code

try:
    llm_inst = init_llm(model_id=MODEL_ID, max_model_len=MAX_MODEL_LEN)
    print(f"✓ Động cơ vLLM khởi tạo thành công: {MODEL_ID}")
except Exception as e:
    print(f"⚠️ Không khởi tạo được GPU vLLM trực tiếp ({e}). Sẽ sử dụng chế độ dự phòng nếu có.")


In [ ]:
# =====================================================================
# CELL 5: CẤU HÌNH CÁC RUNNER (CEGR VÀ CÁC BIẾN THỂ ABLATION)
# =====================================================================
from cegr.synthesis.runner import CegrRunner

def create_runners_for_env(env: DoorKeyEnv) -> Dict[str, CegrRunner]:
    cfg_base = {
        "max_rounds": N_ROUNDS,
        "llm": {"temperature_repair": REPAIR_TEMP, "max_tokens": REPAIR_MAXTOK},
        "scoring_profile": SCORING_PROFILE,
        "keypos_mode": KEYPOS_MODE,
    }
    r_dict = {
        "CEGR":            CegrRunner(env, cfg_base, method_name="CEGR"),
        "CEGR_noClass":    CegrRunner(env, cfg_base, method_name="CEGR_noClass"),
        "CEGR_noFeedback": CegrRunner(env, cfg_base, method_name="CEGR_noFeedback"),
        "CEGR_noPool":     CegrRunner(env, cfg_base, method_name="CEGR_noPool"),
        "CEGR_noAnchors":  CegrRunner(env, cfg_base, method_name="CEGR_noAnchors"),
        "CEGR_noG2":       CegrRunner(env, cfg_base, method_name="CEGR_noG2"),
        "CEGR_noG4":       CegrRunner(env, cfg_base, method_name="CEGR_noG4"),
        "CEGR_noG5":       CegrRunner(env, cfg_base, method_name="CEGR_noG5"),
    }
    return r_dict

env_runners = {env_name: create_runners_for_env(envs[env_name]) for env_name in ENV_NAMES}
print(f"✓ Đã khởi tạo đầy đủ {len(env_runners[ENV_NAMES[0]])} runners cho mỗi môi trường trong {ENV_NAMES}.")


In [ ]:
# =====================================================================
# CELL 6: VÒNG LẶP THỰC NGHIỆM CHÍNH (PHA 2) & GHI LOG JSONL CHUẨN MỤC 2
# =====================================================================
# Khởi tạo file log theo giao thức Mục 2
candidates_log_file = os.path.join(OUT_DIR, "candidates.jsonl")
final_log_file = os.path.join(OUT_DIR, "final.jsonl")

existing_finals = []
if os.path.exists(final_log_file):
    with open(final_log_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                existing_finals.append(json.loads(line))
    print(f"✓ Nạp checkpoint kết quả cuối: {len(existing_finals)} jobs đã có.")

def job_already_done(env: str, arm: str, rep: int, base_id: str, sample_id: int) -> bool:
    for j in existing_finals:
        if (j.get("env") == env and j.get("arm") == arm and 
            j.get("rep") == rep and j.get("base_id") == base_id and 
            j.get("sample_id") == sample_id):
            return True
    return False

# Lấy danh sách manifest
manifest_data = abl_manifest.get("manifest", abl_manifest) if "manifest" in abl_manifest else abl_manifest

print(f">>> BẮT ĐẦU CHẠY THỰC NGHIỆM ĐỐI SÁNH CEGR vs ABLATIONS (3 ENVS) <<<")

for env_name in ENV_NAMES:
    env_inst = envs[env_name]
    target_records = manifest_data[env_name][:N_ABL_REWARDS]
    runners_dict = env_runners[env_name]
    
    print(f"\n===================================================================")
    print(f"===> ĐANG CHẠY MÔI TRƯỜNG: {env_name.upper()} ({len(target_records)} base rewards)")
    print(f"===================================================================")
    
    # 1. Chạy CEGR với R replicate (để tính noise floor)
    for rep in range(N_CEGR_REPLICATES):
        arm_name = "CEGR"
        print(f"\n--- [Env: {env_name}] Arm: {arm_name} | Replicate rep = {rep} ---")
        runner = runners_dict["CEGR"]
        for rec in target_records:
            b_id = rec["base_id"]
            for s_id in range(N_SAMPLES_PER_JOB):
                if job_already_done(env_name, arm_name, rep, b_id, s_id):
                    continue
                
                # Common random number seed cho LLM
                llm_seed = int(hashlib.sha256(f"{rep}_{b_id}_{s_id}".encode()).hexdigest()[:8], 16) % 1_000_000
                res = runner.run_single_job(rec, sample_idx=s_id)
                
                # 1. Ghi log ứng viên từng vòng (candidates.jsonl)
                with open(candidates_log_file, "a", encoding="utf-8") as f_cand:
                    for w in res.get("waves", []):
                        f_cand.write(json.dumps({
                            "run_id": f"{env_name}_{arm_name}_rep{rep}_{b_id}_s{s_id}",
                            "env": env_name, "arm": arm_name, "rep": rep,
                            "base_id": b_id, "sample_id": s_id,
                            "round": w.get("wave"), "round_index_base": 1,
                            "llm_seed": llm_seed,
                            "first_failed_gate": f"G{w.get('failed_gate')}" if w.get("failed_gate") else None,
                            "pool_size_after": w.get("pool_size", 0),
                            "passed_search": bool(w.get("passed", False)),
                        }, ensure_ascii=False) + "\n")
                
                # 2. Ghi log kết quả cuối (final.jsonl)
                final_entry = {
                    "env": env_name, "arm": arm_name, "rep": rep,
                    "base_id": b_id, "sample_id": s_id,
                    "valid": True, "invalid_reason": None,
                    "returned_candidate_round": res.get("rounds_to_pass"),
                    "stop_reason": "pass" if res.get("passed_gates") else "budget",
                    "n_llm_calls": res.get("llm_calls", 0),
                    "k_g5_episodes_search": res.get("rl_episodes", 0),
                    "final_dp_T_ok": bool(res.get("loose_dp", False)),
                    "final_dp_2T_ok": bool(res.get("tier1_safe", False)),
                    "final_subgoal_ok": bool(res.get("culprit_targeted", False)),
                    "final_q_success": float(res.get("rl_success", 0.0)),
                    "final_vf": int(bool(res.get("vf", False))),
                    "dp_hack_flag": bool(res.get("hacked_dp", False)),
                }
                with open(final_log_file, "a", encoding="utf-8") as f_fin:
                    f_fin.write(json.dumps(final_entry, ensure_ascii=False) + "\n")
                existing_finals.append(final_entry)
        print(f"✓ Hoàn thành {arm_name} (rep={rep}) trên {env_name}.")
    
    # 2. Chạy lần lượt các biến thể Ablation
    for arm_name in ABLATION_ARMS:
        print(f"\n--- [Env: {env_name}] Arm Ablation: {arm_name} ---")
        runner = runners_dict[arm_name]
        rep = 0
        for rec in target_records:
            b_id = rec["base_id"]
            for s_id in range(N_SAMPLES_PER_JOB):
                if job_already_done(env_name, arm_name, rep, b_id, s_id):
                    continue
                
                llm_seed = int(hashlib.sha256(f"{rep}_{b_id}_{s_id}".encode()).hexdigest()[:8], 16) % 1_000_000
                res = runner.run_single_job(rec, sample_idx=s_id)
                
                with open(candidates_log_file, "a", encoding="utf-8") as f_cand:
                    for w in res.get("waves", []):
                        f_cand.write(json.dumps({
                            "run_id": f"{env_name}_{arm_name}_rep{rep}_{b_id}_s{s_id}",
                            "env": env_name, "arm": arm_name, "rep": rep,
                            "base_id": b_id, "sample_id": s_id,
                            "round": w.get("wave"), "round_index_base": 1,
                            "llm_seed": llm_seed,
                            "first_failed_gate": f"G{w.get('failed_gate')}" if w.get("failed_gate") else None,
                            "pool_size_after": w.get("pool_size", 0),
                            "passed_search": bool(w.get("passed", False)),
                        }, ensure_ascii=False) + "\n")
                
                final_entry = {
                    "env": env_name, "arm": arm_name, "rep": rep,
                    "base_id": b_id, "sample_id": s_id,
                    "valid": True, "invalid_reason": None,
                    "returned_candidate_round": res.get("rounds_to_pass"),
                    "stop_reason": "pass" if res.get("passed_gates") else "budget",
                    "n_llm_calls": res.get("llm_calls", 0),
                    "k_g5_episodes_search": 0 if arm_name == "CEGR_noG5" else res.get("rl_episodes", 0),
                    "final_dp_T_ok": bool(res.get("loose_dp", False)),
                    "final_dp_2T_ok": bool(res.get("tier1_safe", False)),
                    "final_subgoal_ok": bool(res.get("culprit_targeted", False)),
                    "final_q_success": float(res.get("rl_success", 0.0)),
                    "final_vf": int(bool(res.get("vf", False))),
                    "dp_hack_flag": bool(res.get("hacked_dp", False)),
                }
                with open(final_log_file, "a", encoding="utf-8") as f_fin:
                    f_fin.write(json.dumps(final_entry, ensure_ascii=False) + "\n")
                existing_finals.append(final_entry)
        print(f"✓ Hoàn thành {arm_name} trên {env_name}.")

print(f"\n✓ TẤT CẢ CÁC JOB ĐÃ HOÀN TẤT VÀ LƯU AN TOÀN TẠI: {final_log_file}")


In [ ]:
# =====================================================================
# CELL 7: KIỂM TRA TỰ ĐỘNG TRƯỚC KHI PHÂN TÍCH (SANITY CHECKS MỤC 3)
# =====================================================================
print(">>> ĐANG CHẠY BỘ KIỂM TRA TỰ ĐỘNG (SANITY CHECKS THEO MỤC 3)...")

df_final = pd.read_json(final_log_file, lines=True)
assert not df_final.empty, "File final.jsonl trống!"

# 1. Kiểm tra tính toàn vẹn 3 sample_id cho mỗi ô (env, arm, rep, base_id)
grouped_samples = df_final.groupby(["env", "arm", "rep", "base_id"])["sample_id"].nunique()
incomplete = grouped_samples[grouped_samples < (1 if FAST_SMOKE_TEST else 3)]
if len(incomplete) == 0:
    print("  [PASS] Mọi ô (env, arm, rep, base_id) đều đủ số mẫu theo quy chuẩn.")
else:
    print(f"  [WARN] Có {len(incomplete)} ô chưa đủ số mẫu quy định:")
    print(incomplete.head())

# 2. Kiểm tra tính hợp lệ của VF: final_vf = 1 thì các tiêu chuẩn cơ sở phải thỏa
bad_vf = df_final[(df_final.final_vf == 1) & ((~df_final.final_dp_T_ok) | (~df_final.final_dp_2T_ok))]
assert len(bad_vf) == 0, f"Phát hiện {len(bad_vf)} mẫu có final_vf=1 nhưng rớt DP T hoặc 2T!"
print("  [PASS] Không mẫu nào có final_vf=1 mà rớt các tiêu chuẩn an toàn DP.")

# 3. Kiểm tra k_g5_episodes_search của arm noG5 phải bằng 0
nog5_episodes = df_final[df_final.arm == "CEGR_noG5"]["k_g5_episodes_search"].sum()
assert nog5_episodes == 0, f"Phát hiện arm CEGR_noG5 có sử dụng RL search episodes ({nog5_episodes})!"
print("  [PASS] Arm CEGR_noG5 hoàn toàn không tốn tập RL search episode nào.")

print("✓ Toàn bộ Sanity Checks đã vượt qua thành công! Dữ liệu đạt chuẩn phân tích thống kê.")


In [ ]:
# =====================================================================
# CELL 8: PHÂN TÍCH THỐNG KÊ GHÉP CẶP CHUẨN (MỤC 4 CỦA GIAO THỨC)
# =====================================================================
import numpy as np, pandas as pd

def reward_level(df: pd.DataFrame, env: str, arm: str) -> pd.Series:
    """Tính trung bình final_vf theo base_id (gộp mẫu và các replicate)."""
    d = df[(df.env == env) & (df.arm == arm) & (df.valid)]
    return d.groupby("base_id").final_vf.mean()

def paired_diff(df: pd.DataFrame, env: str, arm_a: str = "CEGR", arm_b: str = "CEGR_noClass") -> Tuple[np.ndarray, List[str]]:
    """Tính chênh lệch ghép cặp ΔVF (pp, dương = có lợi cho CEGR)."""
    a, b = reward_level(df, env, arm_a), reward_level(df, env, arm_b)
    idx = a.index.intersection(b.index)
    return (a[idx] - b[idx]).to_numpy() * 100.0, list(idx)

def perm_p(d: np.ndarray, n_mc: int = 200_000, seed: int = 0) -> float:
    """Kiểm định hoán vị dấu: chính xác 2^n khi n <= 20, Monte Carlo khi n > 20."""
    d = np.asarray(d, float)
    n = len(d)
    if n == 0:
        return 1.0
    obs = abs(d.mean())
    if n <= 20:
        signs = ((np.arange(2**n)[:, None] >> np.arange(n)) & 1) * 2 - 1
        stats = np.abs((signs * d).mean(axis=1))
    else:
        rng = np.random.default_rng(seed)
        signs = rng.choice([-1, 1], size=(n_mc, n))
        stats = np.abs((signs * d).mean(axis=1))
    return float((stats >= obs - 1e-12).mean())

def boot_ci(d: np.ndarray, B: int = 10_000, seed: int = 0) -> Tuple[float, float]:
    """Bootstrap phần trăm trên base reward (10,000 lần lấy mẫu)."""
    d = np.asarray(d, float)
    if len(d) == 0:
        return 0.0, 0.0
    rng = np.random.default_rng(seed)
    m = rng.choice(d, size=(B, len(d)), replace=True).mean(axis=1)
    lo, hi = np.percentile(m, [2.5, 97.5])
    return float(lo), float(hi)

def holm_correction(p_vals: List[float]) -> np.ndarray:
    """Hiệu chỉnh đa giả thuyết Holm."""
    p = np.asarray(p_vals, float)
    m = len(p)
    if m == 0:
        return np.array([])
    order = np.argsort(p)
    adj = np.empty(m)
    run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i])
        adj[i] = min(1.0, run)
    return adj

# 1. ĐO NOISE FLOOR TỪ CÁC REPLICATE CỦA CEGR
print("=== BẢNG 1: NOISE FLOOR TỪ CÁC REPLICATE CỦA CEGR ===")
cegr_df = df_final[(df_final.arm == "CEGR") & (df_final.valid)]
noise_records = []
for env in ENV_NAMES:
    env_sub = cegr_df[cegr_df.env == env]
    reps = sorted(env_sub.rep.unique())
    diffs = []
    if len(reps) >= 2:
        for r1, r2 in itertools.combinations(reps, 2):
            p1 = env_sub[env_sub.rep == r1].groupby("base_id").final_vf.mean()
            p2 = env_sub[env_sub.rep == r2].groupby("base_id").final_vf.mean()
            inter = p1.index.intersection(p2.index)
            if len(inter) > 0:
                d = (p1[inter] - p2[inter]).to_numpy() * 100.0
                diffs.extend(np.abs(d))
    mean_noise = float(np.mean(diffs)) if diffs else 0.0
    max_noise = float(np.max(diffs)) if diffs else 0.0
    noise_records.append({"Environment": env, "Replicates": len(reps), "Mean |ΔVF| (pp)": f"{mean_noise:.2f}", "Max |ΔVF| (pp)": f"{max_noise:.2f}"})

df_noise = pd.DataFrame(noise_records)
print(df_noise.to_string(index=False))

# 2. TÍNH BẢNG IV GHÉP CẶP CHÍNH THỨC
print("\n=== BẢNG IV: ĐỐI SÁNH GHÉP CẶP ABLATION VS CEGR CÙNG ĐỢT CHẠY ===")
table_iv_rows = []
primary_tests = []     # Họ chính: CEGR vs CEGR_noClass across 3 envs
exploratory_tests = []  # Họ thăm dò: Các ablation còn lại

for env in ENV_NAMES:
    # Baseline CEGR trong đợt này
    cegr_vf_series = reward_level(df_final, env, "CEGR")
    cegr_mean_vf = float(cegr_vf_series.mean() * 100.0) if len(cegr_vf_series) > 0 else 0.0
    
    table_iv_rows.append({
        "Env": env, "Arm": "CEGR", "N_rewards": len(cegr_vf_series),
        "VF (%)": f"{cegr_mean_vf:.1f}", "ΔVF (pp)": "-",
        "95% CI": "-", "p (raw)": "-", "p (Holm)": "-"
    })
    
    for arm in ABLATION_ARMS:
        abl_vf_series = reward_level(df_final, env, arm)
        abl_mean_vf = float(abl_vf_series.mean() * 100.0) if len(abl_vf_series) > 0 else 0.0
        
        diff, valid_ids = paired_diff(df_final, env, arm_a="CEGR", arm_b=arm)
        delta_mean = float(diff.mean()) if len(diff) > 0 else 0.0
        p_raw = perm_p(diff)
        ci_lo, ci_hi = boot_ci(diff)
        
        test_meta = {
            "env": env, "arm": arm, "n": len(valid_ids),
            "vf": abl_mean_vf, "delta": delta_mean,
            "ci": f"[{ci_lo:+.1f}, {ci_hi:+.1f}]",
            "p_raw": p_raw,
        }
        if arm == "CEGR_noClass":
            primary_tests.append(test_meta)
        else:
            exploratory_tests.append(test_meta)

# Hiệu chỉnh Holm riêng biệt theo đúng Section 4.2
if primary_tests:
    p_holm_prim = holm_correction([t["p_raw"] for t in primary_tests])
    for idx, t in enumerate(primary_tests):
        t["p_holm"] = p_holm_prim[idx]
        t["family"] = "Primary (H1)"

if exploratory_tests:
    p_holm_expl = holm_correction([t["p_raw"] for t in exploratory_tests])
    for idx, t in enumerate(exploratory_tests):
        t["p_holm"] = p_holm_expl[idx]
        t["family"] = "Exploratory"

all_abl_tests = primary_tests + exploratory_tests
for t in all_abl_tests:
    table_iv_rows.append({
        "Env": t["env"], "Arm": t["arm"], "N_rewards": t["n"],
        "VF (%)": f"{t['vf']:.1f}", "ΔVF (pp)": f"{t['delta']:+.1f}",
        "95% CI": t["ci"], "p (raw)": f"{t['p_raw']:.4f}", "p (Holm)": f"{t['p_holm']:.4f}"
    })

df_table_iv = pd.DataFrame(table_iv_rows)
print(df_table_iv.to_string(index=False))


In [ ]:
# =====================================================================
# CELL 9: XUẤT BIỂU ĐỒ ΔVF KÈM NOISE FLOOR & BẢNG LATEX MỤC 5
# =====================================================================
tex_out_file = os.path.join(OUT_DIR, "table_iv_ablation_paired.tex")
with open(tex_out_file, "w", encoding="utf-8") as f:
    f.write(df_table_iv.to_latex(index=False))
print(f"✓ Đã xuất Bảng IV chuẩn LaTeX tại: {tex_out_file}")

# Vẽ biểu đồ ΔVF với Error bars và vùng Noise floor cho từng môi trường
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)

for idx, env in enumerate(ENV_NAMES):
    ax = axes[idx]
    env_tests = [t for t in all_abl_tests if t["env"] == env]
    if not env_tests:
        continue
    
    arms = [t["arm"].replace("CEGR_", "") for t in env_tests]
    deltas = [t["delta"] for t in env_tests]
    cis = [eval(t["ci"]) for t in env_tests]
    yerr = [[deltas[i] - cis[i][0] for i in range(len(deltas))],
            [cis[i][1] - deltas[i] for i in range(len(deltas))]]
    
    # Lấy noise floor trung bình
    noise_val = float(df_noise[df_noise.Environment == env]["Mean |ΔVF| (pp)"].values[0]) if not df_noise.empty else 0.0
    
    ax.axhspan(-noise_val, noise_val, color='gray', alpha=0.2, label=f'Noise floor (±{noise_val:.1f} pp)')
    ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
    
    bars = ax.bar(arms, deltas, yerr=yerr, capsize=4, color='#1f77b4', alpha=0.85, edgecolor='black')
    ax.set_title(f"Môi trường: {env.upper()}")
    ax.set_ylabel("ΔVF so với CEGR (pp)" if idx == 0 else "")
    ax.set_xticklabels(arms, rotation=45, ha='right')
    ax.legend(loc='upper right', frameon=True)

plt.tight_layout()
fig_path = os.path.join(OUT_DIR, "ablation_paired_diffs.png")
plt.savefig(fig_path, dpi=300)
plt.show()
print(f"✓ Đã lưu hình biểu đồ phân tích tại: {fig_path}")
